# Day 3: measure a gate

Use the drone as a hand-held position probe to measure a circular hoop or rectangular gate in the Lighthouse coordinate frame. We reuse Day 1's labelled radio discovery and Day 2's motor-free logging helpers; the same fitted JSON feeds the race planner.

R2 allows 40 minutes for a shared course survey, plus a 20-minute retry buffer. Share the tracing and laptop roles and keep one independent check per physical gate. After five minutes of connection trouble, retain the attempt and use checked shared data.

Start Jupyter from the project root, select **Crazyflie lectures (.venv)** and run cells with **Shift+Enter**. **Run All** uses a labelled simulated outline. To measure a gate, connect and select `data_source = "live"` below. Results stay in `code/day3/data/<run_id>/`; use a fresh run label for another full session.

In [ ]:
from pathlib import Path
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day3/plan.py").is_file())
sys.path.insert(0, str(root / "code"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from notebook_radio import check_drone
from day3.survey import (
    capture_outline, read_capture, example_outline, fit_survey,
    gate_record, plot_survey, save_survey,
)

run_id = "survey-01"
output = root / "code/day3/data" / run_id
output.mkdir(parents=True, exist_ok=False)
print("Python:", sys.executable)
print("Results:", output)

## Connect

Put the drone on a stable surface, keep clear of propellers and close cfclient. Set `LABEL` to your drone's sticker label, run the cell and confirm your drone blinked.

In [ ]:
LABEL = None  # Your drone's sticker label: 0, 2 or "A".
uri = None  # Clear the previous selection before reconnecting.
if LABEL is not None:
    uri = check_drone(LABEL, cache_directory=output / ".cflib-cache")

## Choose the gate and data

Select `circle` or `rectangle`. Use metres in the agreed right-handed Lighthouse frame, with Z upward. `normal_towards` is a world-frame direction that selects the sign of the fitted normal: choose it to point along the intended forward crossing. Reversing it changes the crossing direction while keeping the same physical plane.

For live data, use a pointer/jig referenced to the estimated drone centre and keep the drone's attitude steady. A cage or deck pressed against the rim introduces an offset. With a fixed attitude, add the known world-frame vector from the logged point to the pointer tip to a **copy** of the fitted centre when building the course; keep the original JSON and correction. Attitude changes would require a sample-by-sample correction.

Choose `saved` to inspect a complete recording from another team, or `simulation` for the example. A failed live attempt remains an error until you explicitly choose another source.

In [ ]:
data_source = "simulation"  # "live", "saved" or "simulation"
shape = "circle"  # Or "rectangle".
gate_name = "hoop-1"
normal_towards = np.array([1.0, 0.0, 0.0])
sample_spacing_m = 0.02
recording_directory = None  # For "saved": Path to a capture with recording.json and positions.csv.
capture_name = "capture-01"  # A new name for each live attempt.
duration_s = 30
preparation_s = 5
points = source = survey = saved_survey = None

## Record the outline

Set `data_source = "live"` above, then run the capture cell. Leave the drone on the surface until **STOP VERIFIED**: the helper sets `stabilizer.stop=1` and verifies a firmware lock and four zero motor outputs. It checks the lock, motor outputs and Lighthouse status throughout recording.

After that message, a **five-second preparation interval** lets you place the pointer on the opening. Sampling for the fit starts at **TRACE STARTED**, then lasts **30 seconds**. Trace the whole inner rim once, slowly and at steady attitude, keeping Lighthouse receivers visible. For a rectangle, include all four edges and corners. Finish the outline near its starting point and wait for **TRACE FINISHED**. One group member operates the notebook while another moves the drone.

The recording ends automatically. Connection, preparation and capture together have a finite deadline (65 seconds with these settings). A lost link, nonzero motor output, invalid Lighthouse status or a gap over 0.5 s rejects the attempt. Partial observations and their failure reason stay in the capture folder.

The firmware stop stays engaged after success, failure or interruption. Place the drone down and power-cycle before normal flight preflight checks.

In [ ]:
captured_directory = None
points = source = survey = saved_survey = None
if data_source == "live":
    if uri is None:
        raise ValueError("Run Connect for the intended drone before recording.")
    captured_directory = capture_outline(
        uri, output / capture_name, ready_for_motor_stop=True,
        duration_s=duration_s, preparation_s=preparation_s,
    )
else:
    print("Live capture skipped. The selection cell uses the source chosen above.")

In [ ]:
points = source = survey = saved_survey = None
if data_source == "live":
    if captured_directory is None:
        raise ValueError("No completed live capture. Retry capture or explicitly select saved data/simulation.")
    points, source = read_capture(captured_directory)
elif data_source == "saved":
    if recording_directory is None:
        raise ValueError("Set recording_directory to a complete shared survey recording.")
    points, source = read_capture(recording_directory)
elif data_source == "simulation":
    points, source = example_outline(shape, seed=0)
else:
    raise ValueError("Choose live, saved or simulation.")
print("Source:", source["source"])
print("Raw positions:", len(points))

## Fit and inspect

The fitter first finds a least-squares plane. It then fits a circle or orthogonal rectangle in that plane. Spatial subsampling reduces the extra weight from places where the pointer paused; the retained positions are saved with the fit.

Inspect the points, fitted boundary and red normal. Plane residuals show departures from the fitted plane; radial or outline RMS/max values describe agreement with the chosen shape. These are fit residuals, not the Day 2 Kalman filter's predicted residuals, and a small value alone does not establish absolute position accuracy.

Cross-check the centre, aperture and normal against manual measurements or an independent second pass. A rectangle's planning radius is half its smaller dimension, the conservative inscribed circular opening. Account separately for the drone/cage envelope, supports, gate thickness and survey/estimation/tracking allowances. A moved gate or Lighthouse frame needs a new check.
After changing the data, shape or normal direction, rerun selection, fitting and saving.

In [ ]:
survey = saved_survey = None  # A failed fit must not leave an older exportable result.
survey = fit_survey(points, shape, normal_towards, source, spacing_m=sample_spacing_m)
record = gate_record(gate_name, survey.fit, survey.points, survey.source)
display({key: value for key, value in record.items() if key != "survey"})
display({key: value for key, value in record["survey"].items()
         if key not in ("points_m", "source")})
print("Retained / raw samples:", len(survey.points), "/", len(points))
figure = plot_survey(survey)
figure.savefig(output / "survey-fit.png", dpi=140)
display(figure)
plt.close(figure)

## Save for the trajectory notebook

After checking the geometry within the group, save one JSON per gate. Existing files are protected; use a new capture/run label to retain another result. The JSON contains the centre, normal, aperture, sampled positions, residuals and source. Simulated exports carry an `example-` name and an explicit simulation label.

Open [02_trajectory.ipynb](02_trajectory.ipynb) and paste the printed `survey_paths` assignment into its gate-selection cell. Set the crossing order there, for example `["hoop-1:+", "hoop-1:-"]` with two laps for four crossings. Keep the original survey when applying any documented pointer offset. A saved survey is an input to planning; the later low-speed check establishes flight readiness.

Keep the JSON, `survey-fit.png`, capture folder and a brief note of the independent check/offset. The command-line alternative and schema are in the [survey reference](../../docs/hoop_survey_challenge.md).

In [ ]:
saved_survey = None
if survey is None:
    raise ValueError("Select data and fit it before saving.")
export_name = ("example-" if survey.source["source"] == "SIMULATED outline" else "") + gate_name
saved_survey = save_survey(survey, output / (export_name + ".json"), export_name)
print("Saved:", saved_survey)
print("survey_paths = [Path(" + repr(str(saved_survey)) + ")]")
print("Gate name for crossing_order:", export_name)